---
# Curso: Data Mining - Semana 6
Docente: Soledad Espezúa (s.espezual@up.edu.pe)

---


<a name='manejoarchivos'></a>
# <font color=blue>Laboratorio: faltantes, codificación y transformación de variables</font>

**Caso:** pacientes, atenciones, medicamentos y detalle de medicación.

Este notebook continúa el laboratorio anterior. La primera parte repasa el tratamiento de faltantes y la codificación; la segunda incorpora valores atípicos, variables derivadas, discretización y escalamiento.


#Sesión 1

## <font color=green>1. Cargar las cuatro tablas</font>

Primero realizamos una limpieza estructural mínima: eliminar filas o columnas completamente vacías y duplicados exactos.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Configuramos un estilo simple para los gráficos.
sns.set_theme(style="whitegrid")

# Cargamos las cuatro tablas del caso clínico.
pacientes = pd.read_excel("Pacientes.xlsx")
atenciones = pd.read_excel("Atenciones.xlsx")
medicamentos = pd.read_excel("Medicamentos.xlsx")
detalle = pd.read_excel("Detalle_Medicacion.xlsx")

In [ ]:
def limpieza_estructural(df):
    # Elimina filas completamente vacías.
    df = df.dropna(how="all")

    # Elimina columnas completamente vacías.
    df = df.dropna(axis=1, how="all")

    # Elimina filas duplicadas exactas.
    df = df.drop_duplicates() # Por defecto, conserva la primera ocurrenci

    # Retorna una copia independiente del DataFrame.
    return df.copy()


# Aplicamos la misma limpieza a las cuatro tablas.
pacientes = limpieza_estructural(pacientes)
atenciones = limpieza_estructural(atenciones)
medicamentos = limpieza_estructural(medicamentos)
detalle = limpieza_estructural(detalle)

# Convertimos fecha a tipo datetime para poder ordenarla y resumirla.
atenciones["fecha"] = pd.to_datetime(atenciones["fecha"])

# Revisamos las dimensiones resultantes.
print("Pacientes:", pacientes.shape)
print("Atenciones:", atenciones.shape)
print("Medicamentos:", medicamentos.shape)
print("Detalle:", detalle.shape)

## <font color=green>2. Reconstruir la base integrada a nivel paciente</font>

Esta sección es un repaso de integración. La unidad de análisis final será:

> **una fila = un paciente**

In [ ]:
tablas = {
    "Pacientes": pacientes,
    "Atenciones": atenciones,
    "Medicamentos": medicamentos,
    "Detalle_Medicacion": detalle
}
for nombre, df in tablas.items():
    print(f"\n--- {nombre} ---")

    # Revisa tipos de datos
    display(df.dtypes.to_frame("tipo"))
    # Revisa valores faltantes
    display(df.isna().sum().to_frame("faltantes"))
    display(df.head(3))

Todavía **no** tratamos `espera_min`. Por eso, `espera_promedio` se calcula con los datos tal como están y puede permanecer faltante para algún paciente.

In [ ]:
# Resumimos todas las atenciones de cada paciente.
resumen_atenciones = (
    atenciones
    .groupby("id_paciente")
    .agg(
        n_atenciones=("id_atencion", "nunique"),
        fecha_primera_atencion=("fecha", "min"),
        fecha_ultima_atencion=("fecha", "max"),
        espera_promedio=("espera_min", "mean")  # mean() ignora NaN por defecto.
    )
    .reset_index()
)

# Enriquecemos cada prescripción con información del medicamento
# y relacionamos la atención con el paciente correspondiente.
detalle_medicacion = (
    detalle
    .merge(
        medicamentos[["id_medicamento", "familia"]],
        on="id_medicamento",
        how="left",
        validate="many_to_one"
    )
    .merge(
        atenciones[["id_atencion", "id_paciente"]],
        on="id_atencion",
        how="left",
        validate="many_to_one"
    )
)

# Resumimos la información de medicación a nivel paciente.
resumen_medicacion = (
    detalle_medicacion
    .groupby("id_paciente")
    .agg(
        n_prescripciones=("id_medicamento", "size"),
        n_medicamentos_distintos=("id_medicamento", "nunique")
    )
    .reset_index()
)

# Integramos los resúmenes con la tabla Pacientes.
# Al finalizar, cada fila representa un paciente.
base_paciente = (
    pacientes
    .merge(resumen_atenciones, on="id_paciente", how="left", validate="one_to_one")
    .merge(resumen_medicacion, on="id_paciente", how="left", validate="one_to_one")
)

display(base_paciente)


**Nota**: Se conserva `dosis_mg` en Detalle_Medicacion
Los faltantes de esa variable se revisan en su granularidad original. Pero en este caso eso como no es conceptualmente claro, porque las dosis pueden corresponder a medicamentos distintos y no necesariamente ser comparables entre sí,  decidimos no imputar automáticamente.

De forma similar `presentacion` de la tabla Medicamentos tampoco la incluiría en base_paciente, porque describe al medicamento, no al paciente.

## <font color=green>3. Detectar faltantes que permanecen en la base integrada</font>

Antes de decidir qué hacer, primero debemos saber **dónde están los faltantes y cuántos son**.

In [ ]:
# Contamos los valores faltantes de cada variable.
faltantes = base_paciente.isna().sum()

# Mostramos solo las variables que tienen al menos un faltante.
print(faltantes[faltantes > 0])

In [ ]:
# Identificamos las columnas que tienen valores faltantes.
columnas_faltantes = (
    faltantes[faltantes > 0]
    .index
    .tolist()
)

# Mostramos solo los pacientes que presentan al menos un faltante.
display(
    base_paciente.loc[
        base_paciente[columnas_faltantes]
        .isna()
        .any(axis=1),

        ["id_paciente"] + columnas_faltantes
    ]
)

**¿Encontrar un valor faltante `NaN` significa que siempre debemos reemplazarlo por algún valor? ¿Qué deberíamos analizar antes de decidir?**

> No. Primero debemos entender la variable, la causa posible del faltante y si una imputación tendría sentido conceptual. En algunos casos se puede imputar; en otros se puede eliminar el registro, excluir la variable o incluso conservar el faltante.



## <font color=green>4. Imputación numérica antes de agregar</font>

Volvemos a la tabla `Atenciones`.

Aquí el faltante pertenece a `espera_min`, cuya granularidad es:

> **una fila = una atención**
### <font color=blue>Caso A: imputar `espera_min` antes de agregar</font>

In [ ]:
# Revisamos primero la distribución de espera_min.
display(atenciones["espera_min"].describe())

# El histograma permite observar la forma de la distribución.
plt.subplots(figsize=(4, 3))
sns.histplot(
    data=atenciones,
    x="espera_min",
    bins=7
)
plt.show()

La distribución presenta cierta simetría y algunos valores altos, por lo que utilizaremos la mediana, que es menos sensible a valores extremos.

In [ ]:
# Calculamos la mediana de espera_min.
mediana_espera = atenciones["espera_min"].median()

# Trabajamos sobre una copia para conservar la variable original.
atenciones_imp = atenciones.copy()

# Creamos una nueva variable imputada.
# Si espera_min tiene un valor, se conserva.
# Si está vacío, se reemplaza por la mediana.
atenciones_imp["espera_min_imp"] = (
    atenciones_imp["espera_min"].fillna(mediana_espera)
)

# Comparamos la variable original y la imputada.
display(atenciones_imp[["id_atencion","id_paciente","espera_min","espera_min_imp"]])

In [ ]:
# Después de tratar el faltante a nivel atención,
# Agregamos a nivel paciente y calculamos el promedio por paciente.

resumen_atenciones_imp = (
    atenciones_imp.groupby("id_paciente")
    .agg(espera_promedio_preimp=("espera_min_imp","mean"))
    .reset_index()
)

display(resumen_atenciones_imp)

**Pregunta para discutir:** ¿Qué variable hemos imputado realmente en este caso: `espera_min` o `espera_promedio`?


> Hemos imputado **`espera_min`**, porque el reemplazo se hizo en la tabla `Atenciones`, donde cada fila representa una atención. Después de imputar, usamos esos valores para calcular `espera_promedio_preimp` a nivel paciente. Por tanto, no hemos imputado directamente `espera_promedio`.


## <font color=green>5. Imputación numérica después de agregar</font>

Ahora trabajamos directamente sobre `base_paciente`.

Aquí el faltante pertenece a `espera_promedio`, cuya granularidad es:

> **una fila = un paciente**

### <font color=blue>Caso B: imputar `espera_promedio` después de agregar</font>


In [ ]:
# Revisamos la variable agregada.
display(base_paciente["espera_promedio"].describe())

# Observamos su distribución.
plt.subplots(figsize=(4, 3))
sns.histplot(
    data=base_paciente,
    x="espera_promedio",
    bins=7
)
plt.show()

La mayor parte de los valores de espera_promedio se concentra aproximadamente entre 20 y 32 minutos. Se observa un valor a los aprox los 42 minutos, lo que genera cierta asimetría hacia la derecha.

Entonces dada la distribución con cierta asimetría y un valor alto, usar la mediana puede ser una medida más robusta para realizar una imputación simple.

In [ ]:
# Calculamos la mediana de espera_promedio.
mediana_promedio = ( base_paciente["espera_promedio"].median())

# Creamos una nueva variable imputada a nivel paciente.
base_paciente["espera_promedio_imp"] = (
    base_paciente["espera_promedio"].fillna(mediana_promedio)
)

# Comparamos la variable original con la nueva.
display(base_paciente[["id_paciente","espera_promedio","espera_promedio_imp"]])

Imputar antes y después de agregar **no es equivalente**:

- antes de agregar se recupera un valor a nivel **atención**;
- después de agregar se recupera una variable a nivel **paciente**.

Para la matriz de esta sesión utilizaremos `espera_promedio_imp`.

## <font color=green>6. ¿Qué pasaría si eliminamos?</font>

No vamos a reemplazar nuestra base principal. Solo evaluaremos cuánto perderíamos al aplicar `dropna()`.


### <font color=blue>CASO A: eliminar a nivel atención.</font>
Elimina filas de atenciones sin tiempo de espera

In [ ]:
# Número de atenciones antes de eliminar.
n_antes = atenciones.shape[0]

# Eliminamos solo las atenciones donde falta espera_min.
# subset indica en qué variable se buscará el faltante.
atenciones_sin_espera = atenciones.dropna(subset=["espera_min"]).copy()

# Número de atenciones que quedan.
n_despues = atenciones_sin_espera.shape[0]

print("Atenciones antes:", n_antes)
print("Atenciones después:", n_despues)
print("Atenciones eliminadas:", n_antes - n_despues)

### <font color=blue>CASO B: eliminar en la base a nivel paciente.</font>
Elimina filas de pacientes con espera_promedio faltante

In [ ]:
# Número de pacientes antes de eliminar.
n_antes = base_paciente.shape[0]

# Eliminamos pacientes cuyo espera_promedio está faltante.
base_reducida = base_paciente.dropna(subset=["espera_promedio"]).copy()

# Número de pacientes que quedan.
n_despues = base_reducida.shape[0]

# Calculamos cuántos pacientes se perderían.
n_eliminados = n_antes - n_despues

# Calculamos el porcentaje que representan.
porcentaje = (n_eliminados / n_antes * 100)

print("Pacientes antes:", n_antes)
print("Pacientes después:", n_despues)
print("Pacientes eliminados:", n_eliminados)
print(f"Porcentaje eliminado: {porcentaje:.1f}%")

**Aprendizaje:**
* eliminar una fila en `Atenciones` elimina una atención el paciente;
* eliminar una fila en `base_paciente` elimina un paciente completo de la unidad de análisis.

## <font color=green> 7. Faltante categórico: `distrito`</font>

Compararemos dos decisiones distintas:

- imputar con la moda;
- conservar explícitamente que el dato es **desconocido**.

In [ ]:
# Revisamos las frecuencias de distrito.
# dropna=False hace visible también la categoría faltante.
base_paciente["distrito"].value_counts(dropna=False)

In [ ]:
# mode() devuelve la moda o las modas de la variable.
modas = base_paciente["distrito"].mode()

display(modas)

moda = modas[0] #selecciona la primera moda.

print("Primera moda:", moda)

In [ ]:
# Opción A: sustituir el faltante por la categoría más frecuente.
base_paciente["distrito_moda"] = base_paciente["distrito"].fillna(moda)

# Opción B: mantener explícitamente que no conocemos el distrito.
base_paciente["distrito_imp"] = base_paciente["distrito"].fillna("Desconocido")

# Comparamos ambas decisiones.
base_paciente[["id_paciente","distrito","distrito_moda","distrito_imp"]]

**¿cuál de las dos alternativas asigna al paciente un distrito sin evidencia directa?**

En este caso preferiremos `"Desconocido"`.

## <font color=green> 8. `dosis_mg`: detectar no significa imputar</font>

`dosis_mg` es numérica, pero antes de imputarla debemos comprobar si su significado y unidad son comparables entre medicamentos y presentaciones.

In [ ]:
# Mostramos las prescripciones donde falta dosis_mg.
display(detalle[detalle["dosis_mg"].isna()])

In [ ]:
# Incorporamos información del medicamento para interpretar el faltante.
detalle_contexto = (
    detalle
    .merge(
        medicamentos[["id_medicamento","nombre","presentacion"]],
        on="id_medicamento",
        how="left",
        validate="many_to_one"
    )
)

# Mostramos solo los registros con dosis faltante
# junto con las variables necesarias para entender su contexto.
display(
    detalle_contexto.loc[
        detalle_contexto["dosis_mg"].isna(),
        ["id_atencion","id_medicamento","nombre","presentacion","dosis_mg", "frecuencia_dia","via"]
    ]
)

### Decisión

**No imputaremos `dosis_mg`** porque no podemos garantizar que la variable y su unidad sean comparables entre medicamentos y presentaciones.

Además, `dosis_mg` no es necesaria para la matriz analítica actual.

## <font color=green>9. Excluir una variable que no utilizaremos</font>

In [ ]:
# Si presentacion no es necesaria para el objetivo actual,
# podemos excluirla explícitamente.

medicamentos_reducidos = medicamentos.drop(columns=["presentacion"]).copy()

display(medicamentos_reducidos)

Recuerde:

- `dropna()` elimina filas o columnas según la presencia de faltantes;
- `drop(columns=[...])` excluye explícitamente una variable.

## <font color=green>10. Codificación binaria</font>

In [ ]:
# Codificamos asegurado:
# No -> 0
# Sí -> 1
base_paciente["asegurado_bin"] = base_paciente["asegurado"].map({"No": 0,"Sí": 1 })

# Codificamos sexo:
# F -> 0
# M -> 1
base_paciente["sexo_bin"] = base_paciente["sexo"].map({"F": 0,"M": 1})

# Comparamos las variables originales y codificadas.
display(
    base_paciente[
        [
            "sexo",
            "sexo_bin",
            "asegurado",
            "asegurado_bin"
        ]
    ]
)

Los valores 0 y 1 son **códigos**. No indican que una categoría sea mayor que la otra.

## <font color=green>11. Codificación nominal con one-hot encoding</font>



Ejemplo de one-hot encoding: usaremos `distrito` únicamente para ilustrar cómo se transforma una variable nominal con varias categorías. En un proyecto real, antes de aplicar esta codificación se debe evaluar si la variable es relevante y cuántas categorías tiene.

En la práctica, distrito puede generar muchas columnas y no necesariamente es una buena característica para el análisis.

In [ ]:
# Creamos columnas indicadoras mediante one-hot encoding.
# La variable nominal distrito se reemplaza por una columna 0/1
# para cada categoría.

# Por defecto get_dummies() no crea una categoría para un valor NaN

base_transformada = pd.get_dummies(
    base_paciente,
    columns=["distrito"],
    prefix="distrito",
    dtype=int
)

# Seleccionamos las columnas generadas para distrito
# identificándolas por el prefijo "distrito_".
columnas_one_hot = [
    columna
    for columna in base_transformada.columns
    if columna.startswith("distrito_")
]

print(columnas_one_hot)

In [ ]:
# Mostramos el resultado de la codificación.
display(base_transformada[["id_paciente"] + columnas_one_hot])

**¿Conviene siempre aplicar one-hot encoding a cualquier variable nominal?**

>No. Depende de su relevancia para el análisis y del número de categorías. Si tiene muchas categorías, el número de columnas puede crecer considerablemente.

## <font color=green>12. Primera matriz analítica</font>

El identificador se conserva para trazabilidad, pero **no se usa como característica analítica**.

El one-hot encoding de `distrito` se mostró como ejemplo de la técnica, pero **no lo incorporaremos a esta primera matriz**. Su inclusión en un proyecto real tendría que justificarse.


In [ ]:
# Conservamos id_paciente por separado para trazabilidad.
id_paciente = base_paciente["id_paciente"].copy()

# Seleccionamos las variables disponibles hasta este punto de la sesión.
columnas_iniciales = [
    "edad",
    "sexo_bin",
    "asegurado_bin",
    "espera_promedio_imp",
    "n_atenciones",
    "n_prescripciones",
    "n_medicamentos_distintos"
]

matriz_inicial = base_paciente[columnas_iniciales].copy()

display(matriz_inicial)


## <font color=green>13. Verificación final</font>

In [ ]:
# Revisamos las dimensiones.
print("Dimensiones:", matriz_inicial.shape)

# Revisamos si quedan faltantes.
print("\nFaltantes por variable:")
print(matriz_inicial.isna().sum())

# Revisamos los tipos de datos.
print("\nTipos de datos:")
print(matriz_inicial.dtypes)


In [ ]:
# Identificamos las columnas numéricas de la matriz inicial.
columnas_numericas = (
    matriz_inicial
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

print(columnas_numericas)
print("Número de pacientes:", id_paciente.shape[0])
print("Número de filas de la matriz:", matriz_inicial.shape[0])


#Sesión 2

---
# <font color=blue>Continuación: outliers, variables derivadas, discretización y escalamiento</font>

En esta parte continuamos trabajando con la misma unidad de análisis:

> **una fila = un paciente**

El objetivo es completar la transformación de los datos y construir una matriz analítica final, sin aplicar técnicas de manera automática.


## <font color=green>14. Detectar valores atípicos</font>

Trabajaremos con `espera_promedio_imp`, porque ya se encuentra a nivel paciente y no tiene valores faltantes.

Primero observamos la distribución. Un valor extremo es un **candidato** a outlier; todavía no significa que sea un error.


In [ ]:
# Observamos la distribución de espera_promedio_imp.
display(base_paciente["espera_promedio_imp"].describe())

plt.subplots(figsize=(4, 2))
sns.boxplot(
    data=base_paciente,
    x="espera_promedio_imp"
)
plt.show()


### Regla IQR

La regla del rango intercuartílico utiliza los cuartiles para definir límites. Los valores que quedan fuera de esos límites se consideran **candidatos a valores atípicos**.


In [ ]:
q1 = base_paciente["espera_promedio_imp"].quantile(0.25)
q3 = base_paciente["espera_promedio_imp"].quantile(0.75)

iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)

candidatos_iqr = base_paciente[
    (base_paciente["espera_promedio_imp"] < limite_inferior) |
    (base_paciente["espera_promedio_imp"] > limite_superior)
]

display(
    candidatos_iqr[
        ["id_paciente", "espera_promedio_imp"]
    ]
)


La regla IQR establece un límite inferior de 10.06 y un límite superior de 40.56. El paciente P011, con espera_promedio_imp = 41.5, queda ligeramente por encima del límite superior, por lo que se identifica como candidato a valor atípico.

**P011** es identificado como candidato a valor atípico por la regla IQR, **pero se conserva porque el valor es plausible y no existe evidencia de error**.

### Comparación con z-score

El z-score indica cuántas desviaciones estándar se encuentra un valor respecto de la media. Usaremos `|z| > 3` como ejemplo de un criterio posible.

Es importante observar que **dos reglas estadísticas pueden identificar candidatos diferentes**.


In [ ]:
analisis_outliers = base_paciente[["id_paciente", "espera_promedio_imp"]].copy()

media = analisis_outliers["espera_promedio_imp"].mean()
desv = analisis_outliers["espera_promedio_imp"].std()

analisis_outliers["z_score"] = (
    (analisis_outliers["espera_promedio_imp"] - media) / desv
)

candidatos_z = analisis_outliers[analisis_outliers["z_score"].abs() > 3]

display(analisis_outliers)
print("Candidatos con |z| > 3:")
display(candidatos_z)

Con la regla **IQR**, P011 fue identificado como candidato a valor atípico porque supera ligeramente el límite superior. Sin embargo, con el criterio de **z-score** \(|z|>3\), ningún caso es identificado como atípico; P011 alcanza aproximadamente \(z=2.05\). Además, en este caso el **z-score** es menos sensible para marcar P011 porque la media y la desviación estándar también se ven afectadas por los valores altos. Por eso **IQR** suele ser más robusto cuando hay asimetría o valores extremos.

**Si la regla IQR identifica un tiempo de espera alto, ¿debemos eliminar automáticamente a ese paciente?**

> No. Primero debemos revisar si el valor es posible y coherente con el contexto. En este caso lo conservaremos porque un tiempo de espera alto puede representar una situación real y no existe evidencia de un error de registro.


## <font color=green>15. Crear variables derivadas</font>

Ahora sí construiremos nuevas características a partir de información existente.

La regla usada para derivar una variable debe respetar la unidad de análisis y tener un significado claro.


### 15.1 ¿El paciente tuvo al menos un reingreso?

`reingreso` está registrado en cada atención. Lo convertimos en un indicador y luego lo resumimos a nivel paciente.


In [ ]:
# 0 = No, 1 = Sí
atenciones["reingreso_bin"] = (
    atenciones["reingreso"]
    .map({"No": 0, "Sí": 1})
)

# max() funciona como para mostrar que "alguna vez ocurrió".
reingreso_paciente = (
    atenciones
    .groupby("id_paciente")
    .agg(
        tuvo_reingreso=("reingreso_bin", "max")
    )
    .reset_index()
)

display(reingreso_paciente)


### 15.2 Tipo de la última atención

La fecha permite ordenar las atenciones. Después conservamos el tipo de la atención más reciente de cada paciente.


In [ ]:
ultima_atencion = (
    atenciones
    .sort_values("fecha")
    .drop_duplicates("id_paciente", keep="last") # nos quedamos con la atención mas reciente
    [["id_paciente", "tipo_atencion"]]
    .rename(
        columns={
            "tipo_atencion": "tipo_ultima_atencion"
        }
    )
)

display(ultima_atencion)

### 15.3 ¿El paciente recibió al menos un antibiótico?

En el laboratorio anterior enriquecimos `Detalle_Medicacion` con la familia del medicamento y con `id_paciente`. Ahora utilizamos esa información para crear una característica nueva.


Se creará una variable derivada a nivel paciente a partir de información que originalmente estaba a nivel de prescripción.


In [ ]:
# Creamos un indicador a nivel de prescripción.
detalle_medicacion["antibiotico_bin"] = (
    detalle_medicacion["familia"] == "Antibiótico"
).astype(int)

# Lo resumimos a nivel paciente.
#un paciente puede tener varias prescripciones,
#se agrupan todas sus filas y se toma el máximo:
antibiotico_paciente = (
    detalle_medicacion
    .groupby("id_paciente")
    .agg(
        recibio_antibiotico=("antibiotico_bin", "max")
    )
    .reset_index()
)

display(antibiotico_paciente)

### 15.4 Días de seguimiento

También podemos derivar una característica temporal a partir de las fechas de primera y última atención.


In [ ]:
base_final = base_paciente.copy()

base_final["dias_seguimiento"] = (
    base_final["fecha_ultima_atencion"]- base_final["fecha_primera_atencion"]
).dt.days

# Incorporamos las nuevas características a la base paciente.
base_final = (
    base_final
    .merge(reingreso_paciente, on="id_paciente", how="left", validate="one_to_one")
    .merge(ultima_atencion, on="id_paciente", how="left", validate="one_to_one")
    .merge(antibiotico_paciente, on="id_paciente", how="left", validate="one_to_one")
)

display(
    base_final[
        [
            "id_paciente",
            "tuvo_reingreso",
            "tipo_ultima_atencion",
            "recibio_antibiotico",
            "dias_seguimiento"
        ]
    ]
)


## <font color=green>16. Discretización</font>

Discretizar significa transformar una variable numérica en categorías definidas por intervalos.

La discretización puede facilitar la interpretación, pero también implica pérdida de información. Los puntos de corte deben estar justificados.


### 16.1 Intervalos definidos con `pd.cut()`

Usaremos la edad como ejemplo. Los límites son definidos explícitamente.


In [ ]:
base_final["grupo_edad"] = pd.cut(
    base_final["edad"],
    bins=[0, 34, 59, float("inf")], #definición de puntos de corte
    labels=["Joven", "Adulto", "Adulto mayor"]
)

display(base_final[["id_paciente", "edad", "grupo_edad"]])

Ahora `grupo_edad` es una variable ordinal porque las categorías tienen un orden natural. Si decidimos incluirla en una matriz numérica, podemos representar ese orden explícitamente.


In [ ]:
# Realizando una codificación ordinal de la variable grupo_edad
orden_edad = {
    "Joven": 0,
    "Adulto": 1,
    "Adulto mayor": 2
}

base_final["grupo_edad_ord"] = (
    base_final["grupo_edad"]
    .map(orden_edad)
    .astype(int)
)

display(
    base_final[
        ["edad", "grupo_edad", "grupo_edad_ord"]
    ]
)


### 16.2 Grupos relativos con `pd.qcut()`

`qcut()` define los límites según la posición de los valores en la distribución. Los grupos tienden a tener cantidades similares de observaciones, pero los puntos de corte dependen de los datos.

Lo usaremos solo como comparación; **no incorporaremos `espera_grupo` a la matriz final**.


In [ ]:
# Dividimos espera_promedio_imp en 3 grupos con cantidades similares de pacientes.
# qcut() define los puntos de corte según la distribución de los datos.

base_final["espera_grupo"] = pd.qcut(
    base_final["espera_promedio_imp"],
    q=3,
    labels=["Baja", "Media", "Alta"]
)

# Mostramos la variable original y el grupo asignado.
display(
    base_final[
        ["id_paciente", "espera_promedio_imp", "espera_grupo"]
    ]
)


## <font color=green>17. Codificar una variable derivada nominal</font>

`tipo_ultima_atencion` es una nueva característica, pero sigue siendo nominal. Por eso, si queremos incorporarla a una matriz numérica, podemos aplicar one-hot encoding.

Este ejemplo es más pertinente para la matriz final que el one-hot de `distrito` mostrado en la clase anterior.


In [ ]:
base_final_transformada = pd.get_dummies(
    base_final,
    columns=["tipo_ultima_atencion"],
    prefix="ult_atencion",
    dtype=int
)

columnas_ultima_atencion = [
    columna
    for columna in base_final_transformada.columns
    if columna.startswith("ult_atencion_")
]

print(columnas_ultima_atencion)

## <font color=green>18. Escalamiento</font>

Variables numéricas con rangos diferentes pueden tener magnitudes muy distintas. Escalar no mejora los datos: cambia su representación para poner determinadas variables en una escala comparable.

No escalaremos indicadores binarios ni columnas one-hot. Compararemos dos alternativas para las variables numéricas seleccionadas.


In [ ]:
# Revisamos únicamente las variables numéricas
# antes de decidir cuáles podrían requerir escalamiento.

variables_numericas = (
    base_final_transformada
    .select_dtypes(include=[np.number])
)
variables_numericas.describe().T[["min", "max", "mean", "std"]]

No todas las variables numéricas necesitan escalarse. Nos interesa principalmente escalar aquellas variables continuas o de conteo cuyos rangos son muy diferentes y que podrían influir de manera desigual en técnicas posteriores basadas en distancia o magnitud.

El `describe()` nos ayuda a observar las escalas, pero la decisión de escalar también depende del tipo de variable y de la técnica que utilizaremos posteriormente.

`grupo_edad_ord` no se escala porque sus valores 0, 1, 2 representan orden, no una magnitud cuantitativa que queramos estandarizar.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Seleccionamos las variables numéricas cuantitativas que podrían
# requerir una escala comparable en técnicas posteriores.
# No incluimos variables binarias, one-hot ni ordinales codificadas.

variables_escalar = [
    "edad",
    "espera_promedio_imp",
    "n_atenciones",
    "n_prescripciones",
    "n_medicamentos_distintos",
    "dias_seguimiento"
]

### 18.1 Normalización Min–Max

Reescala cada variable, normalmente al rango `[0, 1]`. El mínimo pasa a 0 y el máximo a 1. Es sensible a valores extremos porque estos amplían el rango.


In [ ]:
minmax = MinMaxScaler()

base_minmax = base_final_transformada.copy()

base_minmax[variables_escalar] = minmax.fit_transform(
    base_minmax[variables_escalar]
)

display(base_minmax[["id_paciente"] + variables_escalar])

### 18.2 Estandarización z-score

Expresa cada valor en unidades de desviación estándar respecto de la media. Después de la transformación, cada variable tendrá aproximadamente media 0 y desviación estándar 1.

La estandarización **no exige** que los datos sean normales para poder aplicarse.


In [ ]:
estandarizador = StandardScaler()

base_zscore = base_final_transformada.copy()

base_zscore[variables_escalar] = estandarizador.fit_transform(
    base_zscore[variables_escalar]
)

display(
    base_zscore[
        ["id_paciente"] + variables_escalar
    ]
)


### Min–Max vs. z-score

- **Min–Max:** cambia el rango, normalmente a `[0,1]`.
- **z-score:** centra la variable alrededor de 0 y la expresa en desviaciones estándar.
- Ninguna técnica debe aplicarse por rutina: la elección depende del análisis posterior.


## <font color=green>19. Construir la matriz analítica final</font>

La matriz final combina variables originales seleccionadas, variables codificadas y variables derivadas.

Mantendremos `id_paciente` separado para trazabilidad. Tampoco incorporaremos `distrito` ni `espera_grupo` en esta versión final.


In [ ]:
columnas_finales = [
    "edad",
    "sexo_bin",
    "asegurado_bin",
    "espera_promedio_imp",
    "n_atenciones",
    "n_prescripciones",
    "n_medicamentos_distintos",
    "dias_seguimiento",
    "tuvo_reingreso",
    "recibio_antibiotico",
    "grupo_edad_ord"
] + columnas_ultima_atencion

matriz_analitica_final = (
    base_final_transformada[
        columnas_finales
    ]
    .copy()
)

display(matriz_analitica_final)

### Dos representaciones escaladas

Como todavía no hemos elegido una técnica analítica posterior, conservamos la matriz sin escalar y construimos dos versiones alternativas. Así podemos comparar qué cambia con cada transformación.


In [ ]:
# Creamos dos copias de la matriz original.
# Así conservamos una versión sin escalar y generamos
# dos versiones alternativas para comparar.
matriz_final_minmax = matriz_analitica_final.copy()
matriz_final_zscore = matriz_analitica_final.copy()

# Aplicamos Min-Max únicamente a las variables que decidimos escalar.
# Las demás columnas (binarias, one-hot, etc.) se mantienen sin cambios.
matriz_final_minmax[variables_escalar] = (
    minmax.fit_transform(
        matriz_final_minmax[variables_escalar]
    )
)

# Aplicamos estandarización z-score a esas mismas variables.
# Nuevamente, el resto de columnas no se modifica.
matriz_final_zscore[variables_escalar] = (
    estandarizador.fit_transform(
        matriz_final_zscore[variables_escalar]
    )
)

# Comparamos las tres representaciones:
# 1. matriz original sin escalar
# 2. matriz con Min-Max
# 3. matriz con z-score
print("Matriz sin escalar:")
display(matriz_analitica_final.head())

print("Matriz con Min–Max:")
display(matriz_final_minmax.head())

print("Matriz con z-score:")
display(matriz_final_zscore.head())


**¿Cuál de las tres matrices deberíamos usar finalmente?**

>Depende de la técnica de análisis que se utilice después. No se escala por rutina.

**Min–Max** facilita comparar variables dentro de un mismo rango, mientras que **z-score** permite interpretar cada valor respecto de la media y la dispersión de su variable. La elección de una u otra representación dependerá de la técnica analítica posterior.

## <font color=green>20. Verificación final</font>

Antes de utilizar la matriz debemos comprobar que la unidad de análisis se mantiene, que no quedan faltantes inesperados y que las variables tienen la representación prevista.


In [ ]:
print("Filas de la base paciente:", base_final_transformada.shape[0])
print("Filas de la matriz final:", matriz_analitica_final.shape[0])

print("\nFaltantes:")
print(matriz_analitica_final.isna().sum())

print("\nTipos de datos:")
print(matriz_analitica_final.dtypes)

print("\nDimensiones finales:", matriz_analitica_final.shape)


# <font color=blue>Tarea grupal — Aplicación al proyecto</font>

Trabaje ahora con los datos de su proyecto y avance la preparación de su matriz analítica:

1.   **Valores atípicos:** seleccione al menos una variable numérica relevante, detecte posibles outliers y justifique qué decisión tomaría con ellos.
2.   **Variables derivadas:** proponga y construya al menos una variable derivada que tenga sentido para su unidad de análisis y para el objetivo del proyecto.
3.   **Discretización:** evalúe si alguna variable numérica se beneficiaría de ser discretizada. Si la aplica, justifique los puntos de corte o el uso de cuantiles.
4.   **Escalamiento:** revise las variables numéricas de su matriz y determine cuáles deberían escalarse para el análisis posterior. Justifique si usaría Min–Max, z-score o si no requiere escalamiento.
5.   **Matriz analítica:** presente una primera versión actualizada de su matriz, indicando qué variables conserva, transforma, deriva o excluye.


No es obligatorio aplicar todas las técnicas vistas. Lo importante es justificar cuáles tienen sentido para sus datos y su objetivo de análisis.